In [1]:
import pandas as pd
import psycopg2
import pandas as pd
import geopandas as gpd
import requests
from geopandas.tools import sjoin
import time
import datetime
from collections import Counter 
import re
import random
import numpy as np
import unidecode
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from shapely import wkt
import plotly.io as pio


In [2]:
df_patients = pd.read_csv("data/data_cleaned/patients_FR_geocoded_adulte_clinique.csv",sep=";",dtype={'codepost':str}).drop("Unnamed: 0",axis=1)
df_revenus = pd.read_csv("data/insee_revenu/BASE_TD_FILO_DISP_IRIS_2020.csv",sep=";",dtype=str)[["IRIS","DISP_MED20"]]
df_idf = gpd.read_file("../geocodeur/from_hegp/data/zones_geographiques/departements/DEPARTEMENT.shp")

In [3]:
len(df_patients)
#df_revenus.columns

57878

## Préparation des données et jointure nationale 

In [4]:
df_revenus[df_revenus["DISP_MED20"]=="ns"] = np.nan
df_revenus[df_revenus["DISP_MED20"]=="nd"] = np.nan

df_revenus["DISP_MED20"] = df_revenus["DISP_MED20"].astype(float)

quartiles, bords_revenus = pd.qcut(df_revenus["DISP_MED20"], 4, labels=False,retbins=True)

df_revenus["quartile"] = quartiles


df_patients.loc[:,"CODE_IRIS"] = df_patients["CODE_IRIS"].astype(str).str.strip()
df_patients_revenus = df_patients.merge(df_revenus, how="left",left_on="CODE_IRIS",right_on="IRIS")
bords_revenus

len(df_patients_revenus)


57878

### Isolement données IDF et hors IDF

In [5]:
df_patients_revenus = df_patients_revenus.dropna(subset="DISP_MED20")


code_dept_idf = ["75","77","78","91","92","93","94","95"]

df_patients_idf_dept = df_patients_revenus[df_patients_revenus["CODE_DEPT"].isin(code_dept_idf)]

df_patients_idf = df_patients_idf_dept[df_patients_idf_dept["INSEE_REG"]==11]

# print(f"Nb de données non disp nationale {df_patients_revenus["DISP_MED20"].isna().sum()}")
# print(f"Nb de données non disp IDF {df_patients_idf["DISP_MED20"].isna().sum()}")


df_patients_hors_idf_dept = df_patients_revenus[~df_patients_revenus["CODE_DEPT"].isin(code_dept_idf)]
df_patients_hors_idf = df_patients_hors_idf_dept[df_patients_hors_idf_dept["INSEE_REG"]!=11]
df_patients_hors_idf

print(len(df_patients_revenus))
print(len(df_patients_idf))
print(len(df_patients_hors_idf))


df_patients_hors_idf.columns

45429
39809
5620


Index(['Unnamed: 0.2', 'Unnamed: 0.1', 'pseudo_provisoire', 'adresse',
       'codepost', 'nom_commune_postal', 'requete', 'x', 'y', 'score',
       'trust_score', 'street', 'city', 'pc_city', 'ic_city', 'code_dept',
       'dept', 'reg', 'address', 'address_has_num_init',
       'address_has_num_geoloc', 'same_city', 'hostel', 'hosted',
       'date_geoloc', 'geometry', 'CODE_IRIS', 'INSEE_REG', 'CODE_DEPT',
       'patient_sexe', 'date_naissance', 'centre', 'ageaudiag', 'cancernum',
       'date_diag', 'topo_initiale_cim10', 'topo_initialelib', 'patho', 'IRIS',
       'DISP_MED20', 'quartile'],
      dtype='object')

In [6]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0,4,1)]
quartile_count = df_patients_revenus['quartile'].value_counts().sort_index()
colors = ['#cce5ff', '#99ccff', '#66b2ff', '#0073e6']

fig = go.Figure(
    data=[go.Pie(
        labels=labels,
        values=quartile_count.values,
        sort=False,
        hole=.3
        )
    ])
fig.update_layout(title="Répartition en quartiles des revenus disponibles médians associé au domicile de notre patientèle française (en €)")
fig.update_traces(textposition='inside', textinfo='percent',marker=dict(colors=colors))
fig.show()

In [7]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0,4,1)]
quartile_count = df_patients_idf['quartile'].value_counts().sort_index()
colors = ['#cce5ff', '#99ccff', '#66b2ff', '#0073e6']


fig = go.Figure(
    data=[go.Pie(
        labels=labels,
        values=quartile_count.values,
        sort=False,
        hole=.3
        )
    ])
fig.update_layout(title="Répartition en quartiles des revenus disponibles médians associé au domicile de notre patientèle française en IDF (en €)")
fig.update_traces(textposition='inside', textinfo='percent',marker=dict(colors=colors))
fig.show()

In [8]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0,4,1)]
quartile_count = df_patients_hors_idf['quartile'].value_counts().sort_index()
colors = ['#cce5ff', '#99ccff', '#66b2ff', '#0073e6']


fig = go.Figure(
    data=[go.Pie(
        labels=labels,
        values=quartile_count.values,
        sort=False,
        hole=.3
        )
    ])
fig.update_layout(title="Répartition en quartiles des revenus disponibles médians associé au domicile de notre patientèle française hors IDF (en €)")
fig.update_traces(textposition='inside', textinfo='percent',marker=dict(colors=colors))
fig.show()

In [9]:
config = {
  'toImageButtonOptions': {
    'format': 'png', # one of png, svg, jpeg, webp
    'filename': 'custom_image',
    'height': 500,
    'width': 700,
    'scale':6 # Multiply title/legend/axis/canvas sizes by this factor
  }
}


fig.show(config=config)

In [10]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0, 4, 1)]
colors = colors = ['#cce5ff', '#99ccff', '#66b2ff', '#0073e6']

quartile_count_fr = df_patients_revenus['quartile'].value_counts().sort_index()
quartile_count_idf = df_patients_idf['quartile'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf['quartile'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles des revenus  médians associé à la patientèle Curie (hommes + Femmes)",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent',textfont=dict(size=20), marker=dict(colors=colors))
config = {
        'toImageButtonOptions': {
            'format': 'png',  # one of png, svg, jpeg, webp
            'filename': 'revenus_patients',
            'height': 1080,
            'width': 1920,
            'scale': 6  # Multiply title/legend/axis/canvas sizes by this factor
        }
    }

# Show the figure
fig.show(config=config)

In [11]:
#revenus pour les Hommes

labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0, 4, 1)]
colors = colors = ['#cce5ff', '#99ccff', '#66b2ff', '#0073e6']

patients_hommes = df_patients_revenus[df_patients_revenus['patient_sexe']=="M"]
patients_hommes_idf = df_patients_idf[df_patients_idf['patient_sexe']=='M']
patients_hommes_hors_idf = df_patients_hors_idf[df_patients_hors_idf['patient_sexe']=='M']

quartile_count_fr = patients_hommes['quartile'].value_counts().sort_index()
quartile_count_idf = patients_hommes_idf['quartile'].value_counts().sort_index()
quartile_count_hors_idf = patients_hommes_hors_idf['quartile'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles des revenus  médians associé à la patientèle Curie (Hommes)",
                  showlegend=True)

fig.update_traces(textposition='inside', textinfo='percent',textfont=dict(size=20), marker=dict(colors=colors))
config = {
        'toImageButtonOptions': {
            'format': 'png',  # one of png, svg, jpeg, webp
            'filename': 'revenus_patients_hommes',
            'height': 1080,
            'width': 1920,
            'scale': 6  # Multiply title/legend/axis/canvas sizes by this factor
        }
    }

# Show the figure
fig.show(config=config)

In [12]:
#revenus pour les Femmes

labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0, 4, 1)]
colors = colors = ['#cce5ff', '#99ccff', '#66b2ff', '#0073e6']

patients_femmes = df_patients_revenus[df_patients_revenus['patient_sexe']=="F"]
patients_femmes_idf = df_patients_idf[df_patients_idf['patient_sexe']=='F']
patients_femmes_hors_idf = df_patients_hors_idf[df_patients_hors_idf['patient_sexe']=='F']

quartile_count_fr = patients_femmes['quartile'].value_counts().sort_index()
quartile_count_idf = patients_femmes_idf['quartile'].value_counts().sort_index()
quartile_count_hors_idf = patients_femmes_hors_idf['quartile'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles des revenus  médians associé à la patientèle Curie (Femmes)",
                  showlegend=True)

fig.update_traces(textposition='inside', textinfo='percent',textfont=dict(size=20), marker=dict(colors=colors))
config = {
        'toImageButtonOptions': {
            'format': 'png',  # one of png, svg, jpeg, webp
            'filename': 'revenus_patientes_femmes',
            'height': 1080,
            'width': 1920,
            'scale': 6  # Multiply title/legend/axis/canvas sizes by this factor
        }
    }

# Show the figure
fig.show(config=config)

### Répartition des revenus patients curie vs France 

In [13]:
# df_patients_geocoded = pd.read_csv("data/data_cleaned/patients_FR_geocoded_adulte_clinique.csv",sep=";", dtype = {'adresse':str,'codepost':str}).drop(["Unnamed: 0"],axis=1)
# df_revenu = pd.read_csv("data/insee_revenu/BASE_TD_FILO_DISP_IRIS_2020.csv", sep=";")

# df_patients_revenus = df_patients_geocoded.merge(df_revenu[['IRIS','DISP_MED20', 'DISP_GI20']], how='left',left_on="CODE_IRIS", right_on="IRIS" )

# df_pat_describe = pd.DataFrame(df_patients_revenus['DISP_MED20'].describe()).rename(columns={"DISP_MED20":"revenu médian - patients"})
# df_revenu.loc[df_revenu["DISP_MED20"] == "ns", "DISP_MED20"] = np.nan
# df_revenu.loc[df_revenu["DISP_MED20"] == "nd", "DISP_MED20"] = np.nan
# df_revenu = df_revenu.dropna(subset="DISP_MED20")
# df_revenu["DISP_MED20"] = df_revenu["DISP_MED20"].astype(int)
# df_france_describe = pd.DataFrame(df_revenu["DISP_MED20"].describe()).rename(columns={"DISP_MED20":"revenu médian - France"})




# df_patients_revenus.loc[df_patients_revenus["DISP_MED20"] == "ns", "DISP_MED20"] = np.nan
# df_patients_revenus.loc[df_patients_revenus["DISP_MED20"] == "nd", "DISP_MED20"] = np.nan

# df_patients_revenus = df_patients_revenus.dropna(subset="DISP_MED20") 
# df_patients_revenus["DISP_MED20"] = df_patients_revenus["DISP_MED20"].astype(int)



In [14]:
#df_revenu['DISP_MED20'].to_csv('C:/Users/lpokambo/Desktop/PROJET/Projet_Loice/data/sorties/excels/revenu_FR.csv',sep=";")
# df_patients_revenus['DISP_MED20'].to_csv('H:/canc_air/data/socio_eco/revenu_patients.csv',sep=";")

In [15]:
sum_iris = df_patients_revenus[['IRIS','DISP_MED20']].groupby('IRIS').size()


In [16]:
fig = go.Figure()
 
fig.add_trace(go.Violin(y=df_patients_revenus['DISP_MED20'], box_visible=True,
                               meanline_visible=True, line_color='blue', opacity=0.6,
                               x0='Revenu médian - patient curie', name='Patients Curie'))
fig.add_trace(go.Violin(y=df_revenus['DISP_MED20'], box_visible=True, line_color='red',
                               meanline_visible=True, opacity=0.6,
                               x0='Revenu médian - france entière', name='France entière'))


fig.update_layout(title="Répartition des revenus médians des patients atteints de cancer à Curie comparé à ceux de la population française",yaxis_zeroline=False)
fig.show()

## Stats : chomage 

In [17]:
df_chomage = pd.read_csv("data/socio_eco/chomage/base-ic-activite-residents-2020.CSV",sep=";",dtype={'IRIS':str,'P20_ACT1564':float,'P20_CHOM1564':float})[["IRIS","P20_ACT1564","P20_CHOM1564"]]

df_chomage["P20_taux_CHOM1564"]=df_chomage["P20_CHOM1564"]/df_chomage["P20_ACT1564"]*100 

quartile, bords = pd.qcut(df_chomage["P20_taux_CHOM1564"], 4, labels=False,retbins=True)

df_chomage["quartile_chomage"] = quartile



df_patients_revenus_chomage = df_patients_revenus.merge(df_chomage, how="left",left_on="CODE_IRIS",right_on="IRIS")
df_patients_revenus_chomage

df_patients_revenus_chomage = df_patients_revenus_chomage.rename({"quartile":"quartile_revenu",
                                                                "quartile_chomage":"quartile_chomage"},axis=1)




C:\Users\lpokambo\AppData\Local\Temp\ipykernel_6772\4172733218.py:1: DtypeWarning:

Columns (1,3) have mixed types. Specify dtype option on import or set low_memory=False.



In [18]:
df_patients_revenus_chomage = df_patients_revenus_chomage.dropna(subset="P20_taux_CHOM1564")
df_patients_revenus_chomage = df_patients_revenus_chomage.dropna(subset="DISP_MED20")

len(df_patients_revenus_chomage)

45429

In [19]:
df_patients_revenus_chomage = df_patients_revenus_chomage.dropna(subset="P20_taux_CHOM1564")
df_patients_revenus_chomage.to_csv("data/data_cleaned/patients_FR_revenus_chomage.csv",sep=";")

In [20]:
code_dept_idf = ["75","77","78","91","92","93","94","95"]

In [21]:
#revenus et chomage patients sur l'ile de France et par departements d'IDF
df_patients_idf_rev_chom_dept = df_patients_revenus_chomage[df_patients_revenus_chomage["CODE_DEPT"].isin(code_dept_idf)]
df_patients_idf_rev_chom = df_patients_idf_rev_chom_dept[df_patients_idf_rev_chom_dept["INSEE_REG"]==11]

# print(f"Nb de données non disp nationale {df_patients_revenus["DISP_MED20"].isna().sum()}")
# print(f"Nb de données non disp IDF {df_patients_idf["DISP_MED20"].isna().sum()}")

#revenus et chomages patients hors IDF et par departement hors IDF
df_patients_hors_idf_dept = df_patients_revenus_chomage[~df_patients_revenus_chomage["CODE_DEPT"].isin(code_dept_idf)]
df_patients_hors_idf = df_patients_hors_idf_dept[df_patients_hors_idf_dept["INSEE_REG"]!=11]

In [22]:
# chomage patients france
labels = [f"Q{i+1}: {np.round(bords[i],2)} % - {np.round(bords[i+1],2)} %" for i in range(0,len(bords)-1,1)]
quartile_count = df_patients_revenus_chomage['quartile_chomage'].value_counts().sort_index()
colors = ['#0073e6','#66b2ff','#99ccff','#cce5ff']


fig = go.Figure(
    data=[go.Pie(
        labels=labels,
        values=quartile_count.values,
        sort=False,
        hole=.3
        )
    ])
fig.update_layout(title="Répartition en quartiles du taux de chomage associé au domicile de notre patientèle française (en %)")
fig.update_traces(textposition='inside', textinfo='percent',textfont=dict(size=20), marker=dict(colors=colors))
config = {
        'toImageButtonOptions': {
            'format': 'png',  # one of png, svg, jpeg, webp
            'filename': 'chomages',
            'height': 1080,
            'width': 1920,
            'scale': 6  # Multiply title/legend/axis/canvas sizes by this factor
        }
    }

# Show the figure
fig.show(config=config)

In [23]:
# chomage patients idf
labels = [f"Q{i+1}: {np.round(bords[i],2)} % - {np.round(bords[i+1],2)} %" for i in range(0,len(bords)-1,1)]
quartile_count = df_patients_idf_rev_chom['quartile_chomage'].value_counts().sort_index()
colors = ['#0073e6','#66b2ff','#99ccff','#cce5ff']


fig = go.Figure(
    data=[go.Pie(
        labels=labels,
        values=quartile_count.values,
        sort=False,
        hole=.3
        )
    ])
fig.update_layout(title="Répartition en quartiles du taux de chomage associé au domicile de notre patientèle  IDF (en %)")
fig.update_traces(textposition='inside', textinfo='percent',textfont=dict(size=20), marker=dict(colors=colors))
config = {
        'toImageButtonOptions': {
            'format': 'png',  # one of png, svg, jpeg, webp
            'filename': 'revenus_patientes_femmes',
            'height': 1080,
            'width': 1920,
            'scale': 6  # Multiply title/legend/axis/canvas sizes by this factor
        }
    }

# Show the figure
fig.show(config=config)

In [24]:
#chomage patients hors idf
labels = [f"Q{i+1}: {np.round(bords[i],2)} % - {np.round(bords[i+1],2)} %" for i in range(0,len(bords)-1,1)]
quartile_count = df_patients_hors_idf['quartile_chomage'].value_counts().sort_index()
colors = ['#0073e6','#66b2ff','#99ccff','#cce5ff']


fig = go.Figure(
    data=[go.Pie(
        labels=labels,
        values=quartile_count.values,
        sort=False,
        hole=.3
        )
    ])
fig.update_layout(title="Répartition en quartiles du taux de chomage associé au domicile de notre patientèle Hors IDF (en %)")
fig.update_traces(textposition='inside', textinfo='percent',marker=dict(colors=colors))
fig.show()

In [25]:
labels = [f"Q{i+1}: {np.round(bords[i],2)} % - {np.round(bords[i+1],2)} %" for i in range(0,len(bords)-1,1)]
colors = ['#0073e6','#66b2ff','#99ccff','#cce5ff']

quartile_count_fr = df_patients_revenus_chomage['quartile_chomage'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_rev_chom['quartile_chomage'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf['quartile_chomage'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles des revenus  médians associé à la patientèle Curie (hommes + Femmes)",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent',textfont=dict(size=20), marker=dict(colors=colors))
config = {
        'toImageButtonOptions': {
            'format': 'png',  # one of png, svg, jpeg, webp
            'filename': 'chomage',
            'height': 1080,
            'width': 1920,
            'scale': 6  # Multiply title/legend/axis/canvas sizes by this factor
        }
    }

# Show the figure
fig.show(config=config)

In [26]:
#chomage pour les Hommes

labels = [f"Q{i+1}: {np.round(bords[i],2)} % - {np.round(bords[i+1],2)} %" for i in range(0,len(bords)-1,1)]
colors = ['#0073e6','#66b2ff','#99ccff','#cce5ff']

patients_hommes = df_patients_revenus_chomage[df_patients_revenus_chomage['patient_sexe']=="M"]
patients_hommes_idf = df_patients_idf_rev_chom[df_patients_idf_rev_chom['patient_sexe']=='M']
patients_hommes_hors_idf = df_patients_hors_idf[df_patients_hors_idf['patient_sexe']=='M']

quartile_count_fr = patients_hommes['quartile_chomage'].value_counts().sort_index()
quartile_count_idf = patients_hommes_idf['quartile_chomage'].value_counts().sort_index()
quartile_count_hors_idf = patients_hommes_hors_idf['quartile_chomage'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles du taux de chômage associé à la patientèle Curie (Hommes)",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent',textfont=dict(size=20), marker=dict(colors=colors))
config = {
        'toImageButtonOptions': {
            'format': 'png',  # one of png, svg, jpeg, webp
            'filename': 'chomage_hommes',
            'height': 1080,
            'width': 1920,
            'scale': 6  # Multiply title/legend/axis/canvas sizes by this factor
        }
    }

# Show the figure
fig.show(config=config)

In [27]:
#chomage pour les Femmes

labels = [f"Q{i+1}: {np.round(bords[i],2)} % - {np.round(bords[i+1],2)} %" for i in range(0,len(bords)-1,1)]
colors = ['#0073e6','#66b2ff','#99ccff','#cce5ff']

patients_femmes = df_patients_revenus_chomage[df_patients_revenus_chomage['patient_sexe']=="F"]
patients_femmes_idf = df_patients_idf_rev_chom[df_patients_idf_rev_chom['patient_sexe']=='F']
patients_femmes_hors_idf = df_patients_hors_idf[df_patients_hors_idf['patient_sexe']=='F']

quartile_count_fr = patients_femmes['quartile_chomage'].value_counts().sort_index()
quartile_count_idf = patients_femmes_idf['quartile_chomage'].value_counts().sort_index()
quartile_count_hors_idf = patients_femmes_hors_idf['quartile_chomage'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles du taux de chômage associé à la patientèle Curie (Femmes)",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent',textfont=dict(size=20), marker=dict(colors=colors))
config = {
        'toImageButtonOptions': {
            'format': 'png',  # one of png, svg, jpeg, webp
            'filename': 'chomage_femmes',
            'height': 1080,
            'width': 1920,
            'scale': 6  # Multiply title/legend/axis/canvas sizes by this factor
        }
    }

# Show the figure
fig.show(config=config)

## ANALYSE SOCIO-ECONOMIQUE EN FONCTION DE LA PATHOLOGIE

In [28]:
df_patho = df_patients_idf['patho'].unique()
df_patho

array(['Hemato', 'Ophtalmo', 'Uro', 'Sein', 'Gastro', 'Gynéco', 'Sarcome',
       'Thorax', 'ORL', 'Dermato', 'Autre', 'Endocrino', 'Neuro'],
      dtype=object)

### Hemato

In [29]:
df_patients_FR_hemato = df_patients_revenus_chomage[df_patients_revenus_chomage['patho']=='Hemato']

df_patients_idf_hemato = df_patients_idf_rev_chom_dept[df_patients_idf_rev_chom_dept['patho']=='Hemato']

df_patients_hors_idf_dept_hemato = df_patients_hors_idf_dept[df_patients_hors_idf_dept['patho']=='Hemato']


##### 1- En France

In [30]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0,4,1)]
colors = ['#cce5ff', '#99ccff', '#66b2ff', '#0073e6']
quartile_count = df_patients_FR_hemato['quartile_revenu'].value_counts().sort_index()


fig = go.Figure(
    data=[go.Pie(
        labels=labels,
        values=quartile_count.values,
        sort=False,
        hole=.3
        )
    ])
fig.update_layout(title="Répartition en quartiles des revenus disponibles médians associé au domicile de notre patientèle française atteinte de cancer hemato en France (en €)")
fig.update_traces(textposition='inside', textinfo='percent', marker=dict(colors=colors))
fig.show()

In [31]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0,4,1)]
colors = ['#cce5ff', '#99ccff', '#66b2ff', '#0073e6']
quartile_count = df_patients_idf_hemato['quartile_revenu'].value_counts().sort_index()


fig = go.Figure(
    data=[go.Pie(
        labels=labels,
        values=quartile_count.values,
        sort=False,
        hole=.3
        )
    ])
fig.update_layout(title="Répartition en quartiles des revenus disponibles médians associé au domicile de notre patientèle française atteinte de cancer hemato en IDF (en €)")
fig.update_traces(textposition='inside', textinfo='percent',marker=dict(colors=colors))
fig.show()

In [32]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0,4,1)]
colors = ['#cce5ff', '#99ccff', '#66b2ff', '#0073e6']
quartile_count = df_patients_hors_idf_dept_hemato['quartile_revenu'].value_counts().sort_index()


fig = go.Figure(
    data=[go.Pie(
        labels=labels,
        values=quartile_count.values,
        sort=False,
        hole=.3
        )
    ])
fig.update_layout(title="Répartition en quartiles des revenus disponibles médians associé au domicile de notre patientèle française atteinte de cancer hemato hors IDF (en €)")
fig.update_traces(textposition='inside', textinfo='percent',marker=dict(colors=colors))
fig.show()

In [33]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0, 4, 1)]
colors = ['#0073e6','#66b2ff','#99ccff','#cce5ff']

quartile_count_fr = df_patients_FR_hemato['quartile_revenu'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_hemato['quartile_revenu'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_hemato['quartile_revenu'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles des revenus  médians associé a notre patientèle atteinte de cancer hemato",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent',marker=dict(colors=colors))


# Show the figure
fig.show()

In [ ]:
labels = [f"Q{i+1}: {np.round(bords[i],2)} % - {np.round(bords[i+1],2)} %" for i in range(0,len(bords)-1,1)]
colors = ['#0073e6','#66b2ff','#99ccff','#cce5ff']

quartile_count_fr = df_patients_FR_hemato['quartile_chomage'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_hemato['quartile_chomage'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_hemato['quartile_chomage'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles du taux de chomage associé au domicile de notre patientèle française atteinte de cancer Hemato(en %)",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent',marker=dict(colors=colors))

# Show the figure
fig.show()

### Ophtalmo

In [ ]:
df_patients_FR_ophtamo = df_patients_revenus_chomage[df_patients_revenus_chomage['patho']=='Ophtalmo']

df_patients_idf_ophtamo = df_patients_idf_rev_chom_dept[df_patients_idf_rev_chom_dept['patho']=='Ophtalmo']

df_patients_hors_idf_dept_ophtamo = df_patients_hors_idf_dept[df_patients_hors_idf_dept['patho']=='Ophtalmo']

In [ ]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0, 4, 1)]

quartile_count_fr = df_patients_FR_ophtamo['quartile_revenu'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_ophtamo['quartile_revenu'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_ophtamo['quartile_revenu'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles des revenus médians associé a notre patientèle atteinte de cancer ophtamo",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

In [ ]:
labels = [f"Q{i+1}: {np.round(bords[i],2)} % - {np.round(bords[i+1],2)} %" for i in range(0,len(bords)-1,1)]

quartile_count_fr = df_patients_FR_ophtamo['quartile_chomage'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_ophtamo['quartile_chomage'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_ophtamo['quartile_chomage'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles du taux de chomage associé au domicile de notre patientèle atteinte de cancer ophtamo(en %)",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

### Uro

In [ ]:
df_patients_FR_Uro = df_patients_revenus_chomage[df_patients_revenus_chomage['patho']=='Uro']

df_patients_idf_Uro = df_patients_idf_rev_chom_dept[df_patients_idf_rev_chom_dept['patho']=='Uro']

df_patients_hors_idf_dept_Uro = df_patients_hors_idf_dept[df_patients_hors_idf_dept['patho']=='Uro']

In [ ]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0, 4, 1)]

quartile_count_fr = df_patients_FR_Uro['quartile_revenu'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_Uro['quartile_revenu'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_Uro['quartile_revenu'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles des revenus médians associé a notre patientèle atteinte de cancer Uro",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

In [ ]:
labels = [f"Q{i+1}: {np.round(bords[i],2)} % - {np.round(bords[i+1],2)} %" for i in range(0,len(bords)-1,1)]

quartile_count_fr = df_patients_FR_Uro['quartile_chomage'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_Uro['quartile_chomage'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_Uro['quartile_chomage'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles du taux de chomage associé au domicile de notre patientèle atteinte de cancer uro (en %)",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

### Sein

In [ ]:
df_patients_FR_Sein = df_patients_revenus_chomage[df_patients_revenus_chomage['patho']=='Sein']

df_patients_idf_Sein = df_patients_idf_rev_chom_dept[df_patients_idf_rev_chom_dept['patho']=='Sein']

df_patients_hors_idf_dept_Sein = df_patients_hors_idf_dept[df_patients_hors_idf_dept['patho']=='Sein']

In [ ]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0, 4, 1)]

quartile_count_fr = df_patients_FR_Sein['quartile_revenu'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_Sein['quartile_revenu'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_Sein['quartile_revenu'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles des revenus médians associé a notre patientèle atteinte de cancer de Sein",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

In [ ]:
labels = [f"Q{i+1}: {np.round(bords[i],2)} % - {np.round(bords[i+1],2)} %" for i in range(0,len(bords)-1,1)]

quartile_count_fr = df_patients_FR_Sein['quartile_chomage'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_Sein['quartile_chomage'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_Sein['quartile_chomage'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles du taux de chomage associé au domicile de notre patientèle <br> atteinte de cancer sein (en %)",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

### Gastro

In [ ]:
df_patients_FR_Gastro = df_patients_revenus_chomage[df_patients_revenus_chomage['patho']=='Gastro']

df_patients_idf_Gastro = df_patients_idf_rev_chom_dept[df_patients_idf_rev_chom_dept['patho']=='Gastro']

df_patients_hors_idf_dept_Gastro = df_patients_hors_idf_dept[df_patients_hors_idf_dept['patho']=='Gastro']

In [ ]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0, 4, 1)]

quartile_count_fr = df_patients_FR_Gastro['quartile_revenu'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_Gastro['quartile_revenu'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_Gastro['quartile_revenu'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles des revenus médians associé a notre patientèle atteinte de cancer de Gastro",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

In [ ]:
labels = [f"Q{i+1}: {np.round(bords[i],2)} % - {np.round(bords[i+1],2)} %" for i in range(0,len(bords)-1,1)]

quartile_count_fr = df_patients_FR_Gastro['quartile_chomage'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_Gastro['quartile_chomage'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_Gastro['quartile_chomage'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles du taux de chomage associé au domicile de notre patientèle atteinte de cancer Gastro(en %)",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

### Gynéco

In [ ]:
df_patients_FR_Gynéco = df_patients_revenus_chomage[df_patients_revenus_chomage['patho']=='Gynéco']

df_patients_idf_Gynéco = df_patients_idf_rev_chom_dept[df_patients_idf_rev_chom_dept['patho']=='Gynéco']

df_patients_hors_idf_dept_Gynéco= df_patients_hors_idf_dept[df_patients_hors_idf_dept['patho']=='Gynéco']

In [ ]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0, 4, 1)]

quartile_count_fr = df_patients_FR_Gynéco['quartile_revenu'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_Gynéco['quartile_revenu'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_Gynéco['quartile_revenu'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles des revenus médians associé a notre patientèle atteinte de cancer de Gyneco",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

In [ ]:
labels = [f"Q{i+1}: {np.round(bords[i],2)} % - {np.round(bords[i+1],2)} %" for i in range(0,len(bords)-1,1)]

quartile_count_fr = df_patients_FR_Gynéco['quartile_chomage'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_Gynéco['quartile_chomage'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_Gynéco['quartile_chomage'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles du taux de chomage associé au domicile de notre patientèle atteinte de cancer Gyneco(en %)",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

### Sarcome

In [ ]:
df_patients_FR_Sarcome = df_patients_revenus_chomage[df_patients_revenus_chomage['patho']=='Sarcome']

df_patients_idf_Sarcome = df_patients_idf_rev_chom_dept[df_patients_idf_rev_chom_dept['patho']=='Sarcome']

df_patients_hors_idf_dept_Sarcome = df_patients_hors_idf_dept[df_patients_hors_idf_dept['patho']=='Sarcome']

In [ ]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0, 4, 1)]

quartile_count_fr = df_patients_FR_Sarcome['quartile_revenu'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_Sarcome['quartile_revenu'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_Sarcome['quartile_revenu'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles des revenus médians associé a notre patientèle atteinte de cancer de Sarcome",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

In [ ]:
labels = [f"Q{i+1}: {np.round(bords[i],2)} % - {np.round(bords[i+1],2)} %" for i in range(0,len(bords)-1,1)]

quartile_count_fr = df_patients_FR_Sarcome['quartile_chomage'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_Sarcome['quartile_chomage'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_Sarcome['quartile_chomage'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles du taux de chomage associé au domicile de notre patientèle atteinte de cancer Sarcome(en %)",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

### Thorax

In [ ]:
df_patients_FR_Thorax = df_patients_revenus_chomage[df_patients_revenus_chomage['patho']=='Thorax']

df_patients_idf_Thorax = df_patients_idf_rev_chom_dept[df_patients_idf_rev_chom_dept['patho']=='Thorax']

df_patients_hors_idf_dept_Thorax = df_patients_hors_idf_dept[df_patients_hors_idf_dept['patho']=='Thorax']

In [ ]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0, 4, 1)]

quartile_count_fr = df_patients_FR_Thorax['quartile_revenu'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_Thorax['quartile_revenu'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_Thorax['quartile_revenu'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles des revenus médians associé a notre patientèle atteinte de cancer de Thorax",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

In [ ]:
labels = [f"Q{i+1}: {np.round(bords[i],2)} % - {np.round(bords[i+1],2)} %" for i in range(0,len(bords)-1,1)]

quartile_count_fr = df_patients_FR_Thorax['quartile_chomage'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_Sarcome['quartile_chomage'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_Sarcome['quartile_chomage'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles du taux de chomage associé au domicile de notre patientèle atteinte de cancer Thorax(en %)",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

### ORL

In [ ]:
df_patients_FR_ORL = df_patients_revenus_chomage[df_patients_revenus_chomage['patho']=='ORL']

df_patients_idf_ORL = df_patients_idf_rev_chom_dept[df_patients_idf_rev_chom_dept['patho']=='ORL']

df_patients_hors_idf_dept_ORL = df_patients_hors_idf_dept[df_patients_hors_idf_dept['patho']=='ORL']

In [ ]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0, 4, 1)]

quartile_count_fr = df_patients_FR_ORL['quartile_revenu'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_ORL['quartile_revenu'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_ORL['quartile_revenu'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles des revenus médians associé a notre patientèle atteinte de cancer de ORL",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

In [ ]:
labels = [f"Q{i+1}: {np.round(bords[i],2)} % - {np.round(bords[i+1],2)} %" for i in range(0,len(bords)-1,1)]

quartile_count_fr = df_patients_FR_ORL['quartile_chomage'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_ORL['quartile_chomage'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_ORL['quartile_chomage'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles du taux de chomage associé au domicile de notre patientèle atteinte de cancer ORL(en %)",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

### Dermato

In [ ]:
df_patients_FR_Dermato = df_patients_revenus_chomage[df_patients_revenus_chomage['patho']=='Dermato']

df_patients_idf_Dermato = df_patients_idf_rev_chom_dept[df_patients_idf_rev_chom_dept['patho']=='Dermato']

df_patients_hors_idf_dept_Dermato = df_patients_hors_idf_dept[df_patients_hors_idf_dept['patho']=='Dermato']

In [ ]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0, 4, 1)]

quartile_count_fr = df_patients_FR_Dermato['quartile_revenu'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_Dermato['quartile_revenu'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_Dermato['quartile_revenu'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles des revenus médians associé a notre patientèle atteinte de cancer de Dermato",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

In [ ]:
labels = [f"Q{i+1}: {np.round(bords[i],2)} % - {np.round(bords[i+1],2)} %" for i in range(0,len(bords)-1,1)]

quartile_count_fr = df_patients_FR_Dermato['quartile_chomage'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_Dermato['quartile_chomage'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_Dermato['quartile_chomage'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles du taux de chomage associé au domicile <br> de notre patientèle atteinte de cancer Dermato(en %)",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

### Endocrino

In [ ]:
df_patients_FR_Endocrino = df_patients_revenus_chomage[df_patients_revenus_chomage['patho']=='Endocrino']

df_patients_idf_Endocrino = df_patients_idf_rev_chom_dept[df_patients_idf_rev_chom_dept['patho']=='Endocrino']

df_patients_hors_idf_dept_Endocrino = df_patients_hors_idf_dept[df_patients_hors_idf_dept['patho']=='Endocrino']

In [ ]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0, 4, 1)]

quartile_count_fr = df_patients_FR_Endocrino['quartile_revenu'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_Endocrino['quartile_revenu'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_Endocrino['quartile_revenu'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles des revenus médians associé a notre patientèle atteinte de cancer de Endocrino",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

In [ ]:
labels = [f"Q{i+1}: {np.round(bords[i],2)} % - {np.round(bords[i+1],2)} %" for i in range(0,len(bords)-1,1)]

quartile_count_fr = df_patients_FR_Endocrino['quartile_chomage'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_Endocrino['quartile_chomage'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_Endocrino['quartile_chomage'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles du taux de chomage associé au domicile de notre patientèle atteinte de cancer d'endocrino(en %)",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

### Neuro

In [ ]:
df_patients_FR_Neuro = df_patients_revenus_chomage[df_patients_revenus_chomage['patho']=='Neuro']

df_patients_idf_Neuro = df_patients_idf_rev_chom_dept[df_patients_idf_rev_chom_dept['patho']=='Neuro']

df_patients_hors_idf_dept_Neuro = df_patients_hors_idf_dept[df_patients_hors_idf_dept['patho']=='Neuro']

In [ ]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0, 4, 1)]

quartile_count_fr = df_patients_FR_Neuro['quartile_revenu'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_Neuro['quartile_revenu'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_Neuro['quartile_revenu'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles des revenus médians associé a notre patientèle atteinte de cancer Neuro",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

In [ ]:
labels = [f"Q{i+1}: {np.round(bords[i],2)} % - {np.round(bords[i+1],2)} %" for i in range(0,len(bords)-1,1)]

quartile_count_fr = df_patients_FR_Neuro['quartile_chomage'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_Neuro['quartile_chomage'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_Neuro['quartile_chomage'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles du taux de chomage associé au domicile de notre patientèle atteinte de cancer Neuro(en %)",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

### Autre

In [ ]:
df_patients_FR_Autre = df_patients_revenus_chomage[df_patients_revenus_chomage['patho']=='Autre']

df_patients_idf_Autre = df_patients_idf_rev_chom_dept[df_patients_idf_rev_chom_dept['patho']=='Autre']

df_patients_hors_idf_dept_Autre = df_patients_hors_idf_dept[df_patients_hors_idf_dept['patho']=='Autre']

In [ ]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0, 4, 1)]

quartile_count_fr = df_patients_FR_Autre['quartile_revenu'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_Autre['quartile_revenu'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_Autre['quartile_revenu'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles des revenus médians associé a notre patientèle atteinte d'autres types de cancer'",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

In [ ]:
labels = [f"Q{i+1}: {np.round(bords[i],2)} % - {np.round(bords[i+1],2)} %" for i in range(0,len(bords)-1,1)]

quartile_count_fr = df_patients_FR_Autre['quartile_chomage'].value_counts().sort_index()
quartile_count_idf = df_patients_idf_Autre['quartile_chomage'].value_counts().sort_index()
quartile_count_hors_idf = df_patients_hors_idf_dept_Autre['quartile_chomage'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text="Répartition en quartiles du taux de chomage associé au domicile de notre patientèle d'autres types de cancer(en %)",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent')

# Show the figure
fig.show()

In [ ]:
df_chomage = pd.read_csv("data/socio_eco/chomage/base-ic-activite-residents-2020.CSV",sep=";",dtype={'IRIS':str,'P20_ACT1564':float,'P20_CHOM1564':float})[["IRIS","P20_ACT1564","P20_CHOM1564"]]

df_chomage["P20_taux_CHOM1564"]=df_chomage["P20_CHOM1564"]/df_chomage["P20_ACT1564"]*100 

quartile, bords = pd.qcut(df_chomage["P20_taux_CHOM1564"], 4, labels=False,retbins=True)

df_chomage["quartile_chomage"] = quartiles


df_patients_revenus_chomage = df_patients_revenus.merge(df_chomage, how="left",left_on="CODE_IRIS",right_on="IRIS")
df_patients_revenus_chomage

df_patients_revenus_chomage = df_patients_revenus_chomage.rename({"quartile":"quartile_revenu",
                                                                "quartile_chomage":"quartile_chomage"},axis=1)




C:\Users\lpokambo\AppData\Local\Temp\ipykernel_3192\2694183038.py:1: DtypeWarning:

Columns (1,3) have mixed types. Specify dtype option on import or set low_memory=False.

